In [1]:
# Cell 1
import warnings
warnings.filterwarnings('ignore', message='pandas only supports SQLAlchemy connectable', category=UserWarning)

import importlib
import numpy as np
import pandas as pd
from pathlib import Path

import scripts.edop.areas.engine as _eng
importlib.reload(_eng)
from scripts.edop.areas.engine import (
    resolve_polity,
    load_catalog,
    attach_values,
    weighted_quantile,
)
from scripts.shared.db_utils import db_connect

conn = db_connect()
print('connected')

connected


In [2]:
# Cell 2
NSONG_NAME  = 'Northern Song'
NSONG_YEAR  = 1000
NSONG_LEVEL = 6
FROM_YEAR   = 950
TO_YEAR     = 1050

geom_wkt, basin_set, polity_meta = resolve_polity(NSONG_NAME, NSONG_YEAR, NSONG_LEVEL, conn)

print(f"Polity: {NSONG_NAME} @ {NSONG_YEAR} CE, L{NSONG_LEVEL:02d}")
print(f"Basins: {len(basin_set)} | weight sum: {basin_set['weight'].sum():.4f}")
print(f"geom_wkt length: {len(geom_wkt)} chars")
polity_meta

{'id': 4481,
 'name': 'Northern Song',
 'fromyear': 990,
 'toyear': 1017,
 'year': 1000}

---
## Part 1 — LMR per-cell distribution at N Song, year 1000 CE

Query the LMR cells that intersect N Song directly; compute ECC and per-cell `prate` distribution.
Purpose: confirm heterogeneity and record ECC value. Does not gate Part 2.

In [3]:
# Cell 4
# LMR array convention: PostgreSQL index = year_CE + 1  (engine: pg_from = lmr_from + 1)
PG_IDX = NSONG_YEAR + 1  # = 1001 for year 1000 CE

lmr_sql = f"""
WITH buf AS (SELECT ST_GeomFromText('{geom_wkt}', 4326) AS buf_geom),
footprints AS (
    SELECT
        lc.lat,
        CASE WHEN lc.lon > 180 THEN lc.lon - 360 ELSE lc.lon END AS lon_disp,
        lc.prate[{PG_IDX}]  AS prate_val,
        lc.air  [{PG_IDX}]  AS air_val,
        lc.pdsi [{PG_IDX}]  AS pdsi_val,
        ST_MakeEnvelope(
            CASE WHEN lc.lon > 180 THEN lc.lon - 360 ELSE lc.lon END - 1,
            lc.lat - 1,
            CASE WHEN lc.lon > 180 THEN lc.lon - 360 ELSE lc.lon END + 1,
            lc.lat + 1, 4326) AS fp
    FROM temporal.lmr_climate lc
)
SELECT
    f.lat, f.lon_disp,
    f.prate_val * 86400  AS prate_mmday,
    f.air_val            AS air_k,
    f.pdsi_val           AS pdsi,
    ST_Area(ST_Intersection(f.fp, buf.buf_geom)::geography) AS overlap_m2,
    ST_Area(f.fp::geography)                                AS cell_area_m2
FROM footprints f, buf
WHERE ST_Intersects(f.fp, buf.buf_geom)
ORDER BY overlap_m2 DESC
"""
lmr_df = pd.read_sql(lmr_sql, conn)
print(f"Cells intersecting N Song: {len(lmr_df)} (raw count)")

Cells intersecting N Song: 93 (raw count)


In [4]:
# Cell 5
frac = (lmr_df['overlap_m2'] / lmr_df['cell_area_m2']).values
w    = frac / frac.sum()          # normalized
ecc  = float(frac.sum())          # effective cell count

print(f"Raw cell count   : {len(lmr_df)}")
print(f"ECC (w_eff)      : {ecc:.2f}")
print(f"Min overlap frac : {frac.min():.3f}")
print(f"Max overlap frac : {frac.max():.3f}")
print()

for var, col, unit in [('prate', 'prate_mmday', 'mm/day'), ('air', 'air_k', 'K anom'), ('pdsi', 'pdsi', 'dimensionless')]:
    v = lmr_df[col].values.astype(float)
    mean_ = float(np.dot(v, w))
    print(f"{var} @ {NSONG_YEAR} CE  ({unit})")
    print(f"  min={v.min():.3f}  p10={weighted_quantile(v, w, 0.1):.3f}  "
          f"mean={mean_:.3f}  p90={weighted_quantile(v, w, 0.9):.3f}  max={v.max():.3f}")
    print(f"  spread (p90-p10) = {weighted_quantile(v, w, 0.9) - weighted_quantile(v, w, 0.1):.3f}")
    print()

Raw cell count   : 93
ECC (w_eff)      : 65.25
Min overlap frac : 0.000
Max overlap frac : 1.000

prate @ 1000 CE  (mm/day)
  min=-0.068  p10=-0.052  mean=-0.004  p90=0.058  max=0.080
  spread (p90-p10) = 0.110

air @ 1000 CE  (K anom)
  min=-0.503  p10=-0.350  mean=-0.279  p90=-0.218  max=-0.098
  spread (p90-p10) = 0.132

pdsi @ 1000 CE  (dimensionless)
  min=-0.169  p10=0.010  mean=0.164  p90=0.320  max=0.642
  spread (p90-p10) = 0.310



---
## Part 2 — `_weighted_histogram` helper

Draft and test the histogram function across all three substrates before integrating into `engine.py`.

Design:
- **Weighted, not counted.** Bin contributions are summed normalized weights, not raw unit counts.
- **Fixed bin count** (20) so payload size is bounded regardless of unit count.
- **`low_resolution`**: for cells, `w_eff < 5`; for basins, `n_units < 3`.
- Temporal stamp fields (`resolver_year`, `band_t_from`, `band_t_to`) are added by the caller — the
  function does the math only.
- Bins are in native units (0–100 score-space for A–E basins; native units for Band T).

In [5]:
# Cell 7
_N_BINS           = 20
_LOW_RES_CELLS    = 5    # w_eff threshold for hyde_cell / lmr_cell
_LOW_RES_BASINS   = 3    # n_units threshold for basin

def _weighted_histogram(values, raw_weights, unit_type, n_bins=_N_BINS):
    """
    Weighted histogram for the distribution detail block.

    values      : array-like float — per-unit scores (basin) or native values (cells)
    raw_weights : array-like float — fractional coverage (cells: overlap/area; basins: area-weight)
    unit_type   : 'basin' | 'hyde_cell' | 'lmr_cell'
    n_bins      : fixed count; keeps payload bounded regardless of unit count

    Returns dict matching WO21b histogram spec (caller appends temporal stamp fields).
    Returns None if no valid units.
    """
    v = np.asarray(values,      dtype=float)
    w = np.asarray(raw_weights, dtype=float)
    mask = np.isfinite(v) & np.isfinite(w) & (w > 0)
    v, w = v[mask], w[mask]
    if len(v) == 0:
        return None

    w_eff  = float(w.sum())          # ECC for cells; sum-of-area-weights for basins (~1)
    w_norm = w / w_eff               # normalized
    n      = int(len(v))
    mean_  = float(np.dot(v, w_norm))

    vmin, vmax = float(v.min()), float(v.max())
    if vmax == vmin:                 # degenerate: all units identical
        edges = np.linspace(vmin - 0.5, vmin + 0.5, n_bins + 1)
    else:
        edges = np.linspace(vmin, vmax, n_bins + 1)

    # Assign each unit to a bin (clip handles exact-max edge)
    bin_idx    = np.clip(np.digitize(v, edges[1:-1]), 0, n_bins - 1)
    bin_w      = np.zeros(n_bins)
    for i, ww in zip(bin_idx, w_norm):
        bin_w[i] += ww

    low_res = (w_eff < _LOW_RES_CELLS) if unit_type != 'basin' else (n < _LOW_RES_BASINS)

    return {
        'bins':           [round(float(e), 4) for e in edges],
        'weights':        [round(float(ww), 6) for ww in bin_w],
        'n_units':        n,
        'unit_type':      unit_type,
        'low_resolution': low_res,
        'min':            round(vmin, 4),
        'max':            round(vmax, 4),
        'p10':            round(weighted_quantile(v, w, 0.1), 4),
        'p90':            round(weighted_quantile(v, w, 0.9), 4),
        'mean':           round(mean_, 4),
    }

print('_weighted_histogram defined')

_weighted_histogram defined


In [6]:
# Cell 8 — basin test: aridity (api_key); schema_key is 'aridity_index'
catalog  = load_catalog()
matrix_df, _, _ = attach_values(
    basin_set, catalog, conn,
    table='public.basin06',
    view='public.v_basin06_persist_rev1',
)

VAR = 'aridity'   # api_key_s; schema_key is 'aridity_index'
ari_scores  = matrix_df[VAR].dropna()
bs_idx      = basin_set.set_index('hybas_id')
ari_weights = bs_idx.loc[ari_scores.index, 'weight'].values

hist_basin = _weighted_histogram(ari_scores.values, ari_weights, unit_type='basin')
hist_basin.update({'resolver_year': NSONG_YEAR, 'band_t_from': None, 'band_t_to': None})

print(f"Basin histogram — {VAR}")
print(f"  n_units={hist_basin['n_units']}  low_res={hist_basin['low_resolution']}")
print(f"  min={hist_basin['min']}  p10={hist_basin['p10']}  "
      f"mean={hist_basin['mean']}  p90={hist_basin['p90']}  max={hist_basin['max']}")
print(f"  weights sum: {sum(hist_basin['weights']):.6f}  (should be ~1.0)")
print(f"  n_bins populated: {sum(1 for w in hist_basin['weights'] if w > 0)}")

Basin histogram — aridity
  n_units=376  low_res=False
  min=19.2059  p10=39.5767  mean=69.6706  p90=88.5887  max=92.98
  weights sum: 1.000001  (should be ~1.0)
  n_bins populated: 20


In [7]:
# Cell 9 — HYDE test: cropland at year 1000 CE
hyde_sql = f"""
WITH buf AS (SELECT ST_GeomFromText('{geom_wkt}', 4326) AS buf_geom),
epoch AS (
    SELECT step_idx FROM temporal.hyde_times WHERE year_ce = {NSONG_YEAR}
),
cell_overlaps AS (
    SELECT
        hc.area_km2,
        ST_Area(ST_Intersection(hc.geom, buf.buf_geom)::geography) AS overlap_m2,
        hc.cropland[e.step_idx + 1] AS cropland_km2
    FROM temporal.hyde_cells hc, buf, epoch e
    WHERE ST_Intersects(hc.geom, buf.buf_geom)
)
SELECT area_km2, overlap_m2, cropland_km2
FROM cell_overlaps
WHERE overlap_m2 > 0
"""
hyde_df = pd.read_sql(hyde_sql, conn)

frac_h = (hyde_df['overlap_m2'] / (hyde_df['area_km2'] * 1e6)).values
w_eff_h = float(frac_h.sum())
print(f"HYDE cells intersecting N Song: {len(hyde_df)} raw | w_eff={w_eff_h:.1f}")

hist_hyde = _weighted_histogram(
    hyde_df['cropland_km2'].values, frac_h, unit_type='hyde_cell'
)
hist_hyde.update({'resolver_year': NSONG_YEAR, 'band_t_from': FROM_YEAR, 'band_t_to': TO_YEAR})

print(f"\nHYDE histogram — cropland_fraction @ {NSONG_YEAR} CE")
print(f"  n_units={hist_hyde['n_units']}  low_res={hist_hyde['low_resolution']}")
print(f"  min={hist_hyde['min']}  p10={hist_hyde['p10']}  "
      f"mean={hist_hyde['mean']}  p90={hist_hyde['p90']}  max={hist_hyde['max']}")
print(f"  weights sum: {sum(hist_hyde['weights']):.6f}  (should be ~1.0)")
print(f"  n_bins populated: {sum(1 for w in hist_hyde['weights'] if w > 0)}")

HYDE cells intersecting N Song: 37901 raw | w_eff=37375.0

HYDE histogram — cropland_fraction @ 1000 CE
  n_units=37901  low_res=False
  min=0.0  p10=0.0  mean=3.3209  p90=10.6138  max=79.7275
  weights sum: 1.000000  (should be ~1.0)
  n_bins populated: 20


In [8]:
# Cell 10 — LMR test: prate at year 1000 CE  (uses lmr_df from Cell 4)
frac_l = (lmr_df['overlap_m2'] / lmr_df['cell_area_m2']).values
w_eff_l = float(frac_l.sum())

hist_lmr = _weighted_histogram(
    lmr_df['prate_mmday'].values, frac_l, unit_type='lmr_cell'
)
hist_lmr.update({'resolver_year': NSONG_YEAR, 'band_t_from': FROM_YEAR, 'band_t_to': TO_YEAR})

print(f"LMR histogram — prate @ {NSONG_YEAR} CE")
print(f"  n_units={hist_lmr['n_units']}  w_eff={w_eff_l:.1f}  low_res={hist_lmr['low_resolution']}")
print(f"  min={hist_lmr['min']}  p10={hist_lmr['p10']}  "
      f"mean={hist_lmr['mean']}  p90={hist_lmr['p90']}  max={hist_lmr['max']}")
print(f"  weights sum: {sum(hist_lmr['weights']):.6f}  (should be ~1.0)")
print(f"  n_bins populated: {sum(1 for w in hist_lmr['weights'] if w > 0)}")

LMR histogram — prate @ 1000 CE
  n_units=93  w_eff=65.3  low_res=False
  min=-0.0676  p10=-0.052  mean=-0.0042  p90=0.0582  max=0.0801
  weights sum: 1.000002  (should be ~1.0)
  n_bins populated: 20


In [9]:
# Cell 11 — readable dump of all three histogram objects
import json

for label, h in [
    (f'aridity (basin, L06)', hist_basin),
    (f'cropland km² (hyde_cell, {NSONG_YEAR} CE)', hist_hyde),
    (f'prate mm/day (lmr_cell, {NSONG_YEAR} CE)', hist_lmr),
]:
    print(f'=== {label} ===')
    compact = {k: v for k, v in h.items() if k not in ('bins', 'weights')}
    print(json.dumps(compact, indent=2))
    bins = h['bins']
    wts  = h['weights']
    print('  bin edges → weight (non-zero):')
    for i, w in enumerate(wts):
        if w > 0:
            print(f'    [{bins[i]:.3f}, {bins[i+1]:.3f}) → {w:.4f}')
    print()

=== aridity (basin, L06) ===
{
  "n_units": 376,
  "unit_type": "basin",
  "low_resolution": false,
  "min": 19.2059,
  "max": 92.98,
  "p10": 39.5767,
  "p90": 88.5887,
  "mean": 69.6706,
  "resolver_year": 1000,
  "band_t_from": null,
  "band_t_to": null
}
  bin edges → weight (non-zero):
    [19.206, 22.895) → 0.0037
    [22.895, 26.583) → 0.0041
    [26.583, 30.272) → 0.0051
    [30.272, 33.961) → 0.0071
    [33.961, 37.649) → 0.0289
    [37.649, 41.338) → 0.0710
    [41.338, 45.027) → 0.0488
    [45.027, 48.715) → 0.0302
    [48.715, 52.404) → 0.0280
    [52.404, 56.093) → 0.0524
    [56.093, 59.782) → 0.0375
    [59.782, 63.470) → 0.0119
    [63.470, 67.159) → 0.0456
    [67.159, 70.848) → 0.0461
    [70.848, 74.537) → 0.0487
    [74.537, 78.225) → 0.0756
    [78.225, 81.914) → 0.0595
    [81.914, 85.603) → 0.1487
    [85.603, 89.291) → 0.1591
    [89.291, 92.980) → 0.0881

=== cropland km² (hyde_cell, 1000 CE) ===
{
  "n_units": 37901,
  "unit_type": "hyde_cell",
  "low_resoluti

In [10]:
# Cell 12 — gate checks
errors = []

for label, h in [
    ('basin/aridity',     hist_basin),
    ('hyde/cropland',     hist_hyde),
    ('lmr/prate',         hist_lmr),
]:
    w_sum = sum(h['weights'])
    if not (0.9999 < w_sum < 1.0001):
        errors.append(f'{label}: weights sum = {w_sum:.6f} (expected ~1.0)')
    if len(h['bins']) != _N_BINS + 1:
        errors.append(f'{label}: expected {_N_BINS+1} bin edges, got {len(h["bins"])}')
    if len(h['weights']) != _N_BINS:
        errors.append(f'{label}: expected {_N_BINS} weight values, got {len(h["weights"])}')
    if h['n_units'] <= 0:
        errors.append(f'{label}: n_units = {h["n_units"]}')
    if not (h['min'] <= h['p10'] <= h['mean'] <= h['p90'] <= h['max']):
        errors.append(f'{label}: ordering violation min/p10/mean/p90/max')
    if 'resolver_year' not in h:
        errors.append(f'{label}: missing resolver_year')
    if 'band_t_from' not in h or 'band_t_to' not in h:
        errors.append(f'{label}: missing band_t_from/to')

# Sentinel strings must be gone
# (will confirm once engine is edited; skipped here — function under test does not produce them)

if errors:
    for e in errors:
        print('FAIL:', e)
else:
    print('ALL CHECKS PASS')
    print(f'  basin  : n_units={hist_basin["n_units"]}  low_res={hist_basin["low_resolution"]}')
    print(f'  hyde   : n_units={hist_hyde["n_units"]}  low_res={hist_hyde["low_resolution"]}')
    print(f'  lmr    : n_units={hist_lmr["n_units"]}  low_res={hist_lmr["low_resolution"]}')

ALL CHECKS PASS
  basin  : n_units=376  low_res=False
  hyde   : n_units=37901  low_res=False
  lmr    : n_units=93  low_res=False


---
## Engine integration check

Confirm histograms appear in the actual `areal_signature_polygon` payload — not just the standalone function.

In [12]:
# Cell 13 — reload engine, run full payload, spot-check all three substrates
importlib.reload(_eng)
from scripts.edop.areas.engine import areal_signature_polygon

payload = areal_signature_polygon(
    geom_wkt, conn, NSONG_LEVEL,
    bands='ABCDET',
    from_year=FROM_YEAR, to_year=TO_YEAR,
    include_detail=True,
)
rows = payload['rows']
print(f"Total rows: {len(rows)}")

# Basin: find any B1 row and check detail.distribution
b1 = next(r for r in rows if r['method'] == 'area_weighted')
dist_b = b1['detail'].get('distribution')
print(f"\nBasin ({b1['variable']}): distribution present = {dist_b is not None}")
if dist_b:
    print(f"  n_units={dist_b['n_units']}  bins={len(dist_b['bins'])-1}  "
          f"weights_sum={sum(dist_b['weights']):.4f}  low_res={dist_b['low_resolution']}")
    print(f"  sentinel 'reported' present = {'reported' in str(dist_b)}")

# HYDE: find any hyde row and check detail.distribution
hyde_r = next(r for r in rows if r['unit_type'] == 'hyde_cell')
dist_h = hyde_r['detail'].get('distribution')
print(f"\nHYDE ({hyde_r['variable']} @ {hyde_r['year']}): distribution present = {dist_h is not None}")
if dist_h:
    print(f"  n_units={dist_h['n_units']}  bins={len(dist_h['bins'])-1}  "
          f"weights_sum={sum(dist_h['weights']):.4f}  low_res={dist_h['low_resolution']}")
    print(f"  sentinel 'reported' present = {'reported' in str(dist_h)}")
    print(f"  method = {hyde_r['method']}")

# LMR: find any lmr row and check method + detail.distribution
lmr_r = next(r for r in rows if r['unit_type'] == 'lmr_cell')
dist_l = lmr_r['detail'].get('distribution')
print(f"\nLMR ({lmr_r['variable']} @ {lmr_r['year']}): distribution present = {dist_l is not None}")
if dist_l:
    print(f"  n_units={dist_l['n_units']}  bins={len(dist_l['bins'])-1}  "
          f"weights_sum={sum(dist_l['weights']):.4f}  low_res={dist_l['low_resolution']}")
    print(f"  sentinel 'collapsed_subresolution' present = {'collapsed_subresolution' in str(dist_l)}")
    print(f"  method = {lmr_r['method']}")

Total rows: 372

Basin (aridity): distribution present = True
  n_units=376  bins=20  weights_sum=1.0000  low_res=False
  sentinel 'reported' present = False

HYDE (hyde_cropland @ 1000): distribution present = True
  n_units=37901  bins=20  weights_sum=1.0000  low_res=False
  sentinel 'reported' present = False
  method = grid_areal_distribution

LMR (lmr_pdsi @ 950): distribution present = True
  n_units=93  bins=20  weights_sum=1.0000  low_res=False
  sentinel 'collapsed_subresolution' present = False
  method = grid_areal_distribution
